# Gradient Descent: From Five Rows to Real Updates

This notebook shares the same five-row study-hours dataset with the course and reproduces predictions, residuals, MSE, gradients, one update, learning-rate paths, and real batches.

In [ ]:
from pathlib import Path
import csv, math
import numpy as np

rows = list(csv.DictReader(Path('study-score.csv').open(encoding='utf-8')))
x = np.array([float(row['study_hours']) for row in rows])
y = np.array([float(row['score']) for row in rows])
print('x =', x.astype(int).tolist())
print('y =', y.astype(int).tolist())

x = [1, 2, 3, 4, 5]
y = [52, 59, 65, 72, 78]


In [ ]:
w, b = 6.0, 47.0
y_hat = w*x + b
r = y-y_hat
mse = np.mean(r**2)
print('prediction =', y_hat.astype(int).tolist())
print('residual =', r.astype(int).tolist())
print(f'MSE = {mse:.6f}')

prediction = [53, 59, 65, 71, 77]
residual = [-1, 0, 0, 1, 1]
MSE = 0.600000


In [ ]:
weight_grid = np.linspace(4, 9, 31)
bias_grid = np.linspace(38, 54, 33)
surface = np.array([[np.mean((y-(candidate_w*x+candidate_b))**2) for candidate_w in weight_grid] for candidate_b in bias_grid])
minimum = np.unravel_index(np.argmin(surface), surface.shape)
print('grid shape =', surface.shape)
print('best grid point =', float(weight_grid[minimum[1]]), float(bias_grid[minimum[0]]))

grid shape = (33, 31)
best grid point = 6.5 45.5


In [ ]:
dw = -(2/len(x))*np.sum(x*r)
db = -(2/len(x))*np.sum(r)
eta = 0.02
new_w, new_b = w-eta*dw, b-eta*db
new_mse = np.mean((y-(new_w*x+new_b))**2)
print(f'gradient = ({dw:.6f}, {db:.6f})')
print(f'updated = ({new_w:.6f}, {new_b:.6f})')
print(f'new MSE = {new_mse:.6f}')

gradient = (-3.200000, -0.400000)
updated = (6.064000, 47.008000)
new MSE = 0.440192


In [ ]:
def run(rate, steps=12):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(steps):
        residual = y-(w*x+b)
        w -= rate * (-(2/len(x))*np.sum(x*residual))
        b -= rate * (-(2/len(x))*np.sum(residual))
        losses.append(float(np.mean((y-(w*x+b))**2)))
    return losses

for rate in [0.002, 0.02, 0.08, 0.3]:
    values = run(rate)
    print(rate, [round(value, 4) for value in values[:4]], '->', f'{values[-1]:.4g}')

0.002 [3962.5121, 3623.8634, 3316.4713, 3037.4476] -> 1558
0.02 [1417.6143, 605.0455, 376.7229, 310.5542] -> 257.2
0.08 [3502.2284, 2835.3177, 2301.2658, 1873.2722] -> 423.1
0.3 [150240.8112, 5578950.6052, 207488330.9222, 7717018456.3698] -> 2.826e+22


In [ ]:
def terrain_values(name, x_value, y_value):
    if name == 'saddle': return x_value**2-y_value**2
    if name == 'rosenbrock': return (1-x_value)**2+20*(y_value-x_value**2)**2
    if name == 'multi-well': return .08*(x_value**2+y_value**2)+np.sin(2*x_value)*np.sin(2*y_value)
    return .08*(x_value+y_value)**2+1.2*(x_value-y_value)**2

for name in ['tilted-ravine', 'rosenbrock', 'saddle', 'multi-well']:
    print(name, round(float(terrain_values(name, 1.0, -0.5)), 6))

tilted-ravine 2.72
rosenbrock 45.0
saddle 0.75
multi-well -0.665147


In [ ]:
def lcg_order(seed):
    state = seed & 0xffffffff
    order = list(range(len(x)))
    for index in range(len(order)-1, 0, -1):
        state = (1664525*state+1013904223) & 0xffffffff
        target = int((state/2**32)*(index+1))
        order[index], order[target] = order[target], order[index]
    return order

order = lcg_order(2801)
mini_batches = [order[offset:offset+2] for offset in range(0, len(order), 2)]
print('mini-batch sample ids =', [[rows[index]['sample_id'] for index in batch] for batch in mini_batches])
print('last mini-batch size =', len(mini_batches[-1]))

mini-batch sample ids = [['s4', 's1'], ['s3', 's5'], ['s2']]
last mini-batch size = 1
